In [ ]:
import pandas as pd
import unicodedata

def to_camel_case(text):
    if pd.isna(text):
        return ""

    # Convertir a string y minúsculas
    text = str(text).strip().lower()

    # Quitar acentos
    text = unicodedata.normalize("NFKD", text)
    text = text.encode("ascii", "ignore").decode("utf-8")

    # Reemplazar caracteres especiales
    text = text.replace("-", " ")
    text = text.replace("_", " ")
    text = text.replace(",", " ")
    text = text.replace(".", " ")
    text = text.replace("(", " ")
    text = text.replace(")", " ")

    # Crear CamelCase
    words = text.split()
    text = "".join(word.capitalize() for word in words)

    return text


df = pd.read_csv("results.csv")

# Eliminar filas completamente vacias
df = df.dropna(how="all")

# Eliminar filas con datos importantes faltantes
df = df.dropna(subset=[
    "date",
    "home_team",
    "away_team",
    "home_score",
    "away_score",
    "neutral"
])

# Columnas de texto a normalizar
text_columns = [
    "home_team",
    "away_team",
    "tournament",
    "city",
    "country"
]

# Aplicar CamelCase
for col in text_columns:
    df[col] = df[col].apply(to_camel_case)

# Convertir fecha
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# Eliminar fechas inválidas
df = df.dropna(subset=["date"])

# Convertir goles a enteros
df["home_score"] = df["home_score"].astype(int)
df["away_score"] = df["away_score"].astype(int)

# Convertir neutral a 0 y 1
df["neutral"] = df["neutral"].astype(str).str.lower().map({
    "true": 1,
    "false": 0
})

# Crear columna winner
def get_winner(row):
    if row["home_score"] > row["away_score"]:
        return row["home_team"]
    elif row["away_score"] > row["home_score"]:
        return row["away_team"]
    else:
        return "Draw"

df["winner"] = df.apply(get_winner, axis=1)

# Crear columna result

def get_result(row):
    if row["home_score"] > row["away_score"]:
        return "home_win"
    elif row["away_score"] > row["home_score"]:
        return "away_win"
    else:
        return "draw"

df["result"] = df.apply(get_result, axis=1)


df = df.sort_values("date").reset_index(drop=True)


df.to_csv("cleanedData.csv", index=False)

print(df.shape)